In [1]:
import os
import re
import pandas as pd
from datasets import load_dataset

In [3]:
dataset = load_dataset("ucberkeley-dlab/measuring-hate-speech")
df = dataset["train"].to_pandas()

print("Raw shape:", df.shape)

Raw shape: (135556, 143)


In [4]:
core_cols = [
    "comment_id",
    "annotator_id",
    "text",
]

survey_item_cols = [
    "sentiment",
    "respect",
    "insult",
    "humiliate",
    "status",
    "dehumanize",
    "violence",
    "genocide",
    "attack_defend",
    "hatespeech"
]

annotator_gender_cols = [
    "annotator_gender",
    "annotator_trans",
    "annotator_gender_men",
    "annotator_gender_women",
    "annotator_gender_non_binary",
    "annotator_gender_prefer_not_to_say",
    "annotator_gender_self_describe",
    "annotator_transgender",
    "annotator_cisgender",
    "annotator_transgender_prefer_not_to_say"
]

annotator_ideology_cols = [
    "annotator_ideology",
    "annotator_ideology_extremeley_conservative",
    "annotator_ideology_conservative",
    "annotator_ideology_slightly_conservative",
    "annotator_ideology_neutral",
    "annotator_ideology_slightly_liberal",
    "annotator_ideology_liberal",
    "annotator_ideology_extremeley_liberal",
    "annotator_ideology_no_opinion"
]

target_gender_cols = [
    "target_gender",
    "target_gender_men",
    "target_gender_non_binary",
    "target_gender_transgender_men",
    "target_gender_transgender_unspecified",
    "target_gender_transgender_women",
    "target_gender_women",
    "target_gender_other"
]

target_origin_cols = [
    "target_origin",
    "target_origin_immigrant",
    "target_origin_migrant_worker",
    "target_origin_specific_country",
    "target_origin_undocumented",
    "target_origin_other"
]

selected_cols = (
    core_cols
    + survey_item_cols
    + annotator_gender_cols
    + annotator_ideology_cols
    + target_gender_cols
    + target_origin_cols
)

len(selected_cols)

46

In [5]:
mhs = df[selected_cols].copy()

print("Selected shape:", mhs.shape)
mhs.head()

Selected shape: (135556, 46)


,comment_id,annotator_id,text,sentiment,respect,insult,humiliate,status,dehumanize,violence,...,target_gender_transgender_unspecified,target_gender_transgender_women,target_gender_women,target_gender_other,target_origin,target_origin_immigrant,target_origin_migrant_worker,target_origin_specific_country,target_origin_undocumented,target_origin_other
0,47777,10873,Yes indeed. She sort of reminds me of the elde...,0.0,0.0,0.0,0.0,2.0,0.0,0.0,...,False,False,False,False,False,False,False,False,False,False
1,39773,2790,The trans women reading this tweet right now i...,0.0,0.0,0.0,0.0,2.0,0.0,0.0,...,False,False,False,False,False,False,False,False,False,False
2,47101,3379,Question: These 4 broads who criticize America...,4.0,4.0,4.0,4.0,4.0,4.0,0.0,...,False,False,False,False,True,True,False,False,False,False
3,43625,7365,It is about time for all illegals to go back t...,2.0,3.0,2.0,1.0,2.0,0.0,0.0,...,False,False,False,False,True,False,False,False,True,False
4,12538,488,For starters bend over the one in pink and kic...,4.0,4.0,4.0,4.0,4.0,4.0,4.0,...,False,False,True,False,False,False,False,False,False,False


In [6]:
def clean_text(text):
    if pd.isna(text):
        return None
    text = str(text)
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text

mhs["text_original"] = mhs["text"]
mhs["text_clean"] = mhs["text"].apply(clean_text)

mhs = mhs.drop(columns=["text"])
mhs = mhs.dropna(subset=["text_clean"])
mhs = mhs[mhs["text_clean"].str.len() > 0]

print("After text cleaning:", mhs.shape)

After text cleaning: (135556, 47)


In [7]:
print("Unique comments:", mhs["comment_id"].nunique())
print("Unique annotators:", mhs["annotator_id"].nunique())
print("Total annotation rows:", len(mhs))

print("\nDuplicate comment IDs are expected:")
print("Duplicate comment_id count:", mhs["comment_id"].duplicated().sum())

Unique comments: 39565
Unique annotators: 7912
Total annotation rows: 135556

Duplicate comment IDs are expected:
Duplicate comment_id count: 95991


In [8]:
os.makedirs("../data/processed", exist_ok=True)

output_path = "../data/processed/mhs_annotations_clean.csv"

mhs.to_csv(output_path, index=False)

print(f"Saved cleaned annotation-level dataset to: {output_path}")
print("Final shape:", mhs.shape)

Saved cleaned annotation-level dataset to: ../data/processed/mhs_annotations_clean.csv
Final shape: (135556, 47)
